# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides an example workflow for loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure mlcroissant library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is an object, not a dict

# Print metadata summary
print(f"Dataset: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")
print(f"Date Published: {metadata.datePublished}\n")
print(f"Authors: {[a for a in metadata.author] if hasattr(metadata, 'author') else 'N/A'}\n")
if hasattr(metadata, 'dataBiases'):
    print(f"Data Biases: {metadata.dataBiases}\n")
if hasattr(metadata, 'dataLimitations'):
    print(f"Data Limitations: {metadata.dataLimitations}\n")


## 2. Data Overview

Explore available record sets, their `@id`s, fields, and columns. All references to data structures use their `@id` according to the Croissant standard.

Below, we print the available record sets and for each, its fields and columns, all referenced by their `@id`s.

In [ ]:
# List all record set @ids in the dataset
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name', '<no name>')}")
    # List the field @ids for the record set
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("    Fields by @id:")
        for f in fields:
            print(f"      * {f.get('@id', str(f))}")
    # List the column @ids for the record set (if any)
    if 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print("    Columns by @id:")
        for col in columns:
            print(f"      + {col.get('@id', str(col))}")
    print()

# For convenience, collect all record set @ids for use below
record_set_ids = [rs['@id'] for rs in record_sets]
if not record_set_ids:
    print('No record sets found in the dataset schema.')


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s printed above.

In case multiple record sets exist, you can iterate through all. Here, we load data for all found record set `@id`s.

In [ ]:
if record_set_ids:
    dataframes = {}
    print('Loading records for each record set...')
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for record set '@id': {rs_id}")
            print(f"Columns: {list(df.columns)}")
        except Exception as e:
            print(f"Failed to load records for record set '@id': {rs_id} --> {e}")

    # Display head of the first record set found
    chosen_rs_id = record_set_ids[0]
    if chosen_rs_id in dataframes:
        print(f"\nFirst 5 records for record set '@id': {chosen_rs_id}")
        display(dataframes[chosen_rs_id].head())
else:
    print('No record sets available to extract data from.')

## 4. Exploratory Data Analysis (EDA)

Next, we demonstrate some standard data cleaning and exploration operations: filtering, normalization, and grouping.

For demonstration, we select a numeric field (by `@id`) and, if available, a categorical/group field for grouping.

In [ ]:
# Please adjust the variables below to match an actual numeric and group field @id for your dataset.
if record_set_ids and dataframes:
    # Use the first record set for demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Columns for record set '@id': {record_set_id}:")
    print(df.columns.tolist())
    
    # Try to automatically guess a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        print(f"\nUsing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_float_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing up to 5 records):")
        print(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized field '{numeric_field}' (first 5 rows):")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by the first non-numeric column, if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}' (showing first 5 groups):")
            print(grouped_df.head())
        else:
            print('No non-numeric field available for grouping in this record set.')
    else:
        print('No numeric fields found for EDA demonstration.')
else:
    print('No data available for EDA.')

## 5. Visualization

Visualize numeric data distributions or relationships between fields.

We plot a histogram and, if possible, a boxplot for the numeric field used above.

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and dataframes:
    df = dataframes[record_set_id]
    if numeric_field and numeric_field in df.columns:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        df[numeric_field].hist(bins=20)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        # Boxplot
        plt.subplot(1,2,2)
        df.boxplot(column=numeric_field)
        plt.title(f"Boxplot of {numeric_field}")
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric field available for visualization.')
else:
    print('No data available for visualization.')

## 6. Conclusion

This notebook demonstrated how to load a FAIR² dataset from a Croissant schema with `mlcroissant`, inspect metadata, enumerate record sets, and perform basic exploratory analysis using Pandas. For your own analysis, refer to available `@id`s to access fields, columns, or record sets of interest, and further customize the workflow for your research or application needs.

To learn more, visit:
- [mlcroissant documentation](https://github.com/mlcommons/croissant)
- [FAIR² dataset package](https://doi.org/10.71728/senscience.y7m0-f273)